<a href="https://colab.research.google.com/github/ypg1um-arch/SAU_ML_TASKS/blob/main/Task_3B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np

class LinearRegressionNormalEquation:
    def __init__(self, add_bias=True):
        self.add_bias = add_bias
        self.weights = None

    def fit(self, X, y):
        """
        Solves for weights analytically using the Normal Equation.
        """
        X = np.asarray(X)
        y = np.asarray(y)

        #Add intercept/bias column of 1s if requested
        if self.add_bias:
            bias_column = np.ones((X.shape[0], 1))
            X = np.hstack([bias_column, X])

        #Apply Normal Equation formula: W = (X^T * X)^(-1) * X^T * y
        #We add a tiny ridge penalty (1e-10) to the diagonal to prevent mathematical crash if features are collinear.
        XT_X = X.T @ X
        stabilizer = np.eye(XT_X.shape[0]) * 1e-10

        self.weights = np.linalg.inv(XT_X + stabilizer) @ X.T @ y

    def predict(self, X):
        """Generates predictions for target values."""
        X = np.asarray(X)

        if self.add_bias:
            bias_column = np.ones((X.shape[0], 1))
            X = np.hstack([bias_column, X])

        return X @ self.weights
if __name__ == "__main__":
    #Generate 100 sample data points with 2 distinct features
    np.random.seed(123)
    X_samples = np.random.rand(100, 2) * 10

    #Target formula used: y = 3.5 + 1.5*(Feature_1) - 2.0*(Feature_2) + Noise
    y_samples = 3.5 + 1.5 * X_samples[:, 0] - 2.0 * X_samples[:, 1] + np.random.normal(0, 0.1, 100)

    #Initialize and train our analytical model
    regressor = LinearRegressionNormalEquation(add_bias=True)
    regressor.fit(X_samples, y_samples)

    #Inspect extraction weights
    print("--- Learned Parameters ---")
    print(f"Calculated Intercept (Bias): {regressor.weights[0]:.4f} (Expected: ~3.5)")
    print(f"Calculated Feature 1 Weight: {regressor.weights[1]:.4f} (Expected: ~1.5)")
    print(f"Calculated Feature 2 Weight: {regressor.weights[2]:.4f} (Expected: ~-2.0)")

    #Test prediction vector performance
    X_test = np.array([[5.0, 3.0]])
    prediction = regressor.predict(X_test)
    print(f"\nPrediction for test features [5.0, 3.0]: {prediction[0]:.4f}")


--- Learned Parameters ---
Calculated Intercept (Bias): 3.4444 (Expected: ~3.5)
Calculated Feature 1 Weight: 1.5072 (Expected: ~1.5)
Calculated Feature 2 Weight: -1.9974 (Expected: ~-2.0)

Prediction for test features [5.0, 3.0]: 4.9885


In [7]:
import numpy as np

class RegularizedLinearRegression:
    def __init__(self, alpha=1.0, penalty='ridge', max_iter=1000, tol=1e-4):
        """
        Regularized Linear Regression from scratch.
        alpha: Regularization strength (must be positive).
        penalty: 'ridge' (L2) or 'lasso' (L1).
        """
        self.alpha = alpha
        self.penalty = penalty.lower()
        self.max_iter = max_iter
        self.tol = tol
        self.weights = None
        self.intercept = 0.0

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).flatten()
        n_samples, n_features = X.shape

        if self.penalty == 'ridge':
            #RIDGE (L2) CLOSED-FORM SOLUTION
            #Formula: w = (X^T * X + alpha * I)^(-1) * X^T * y
            #Center the data to cleanly separate intercept from weights
            X_mean = np.mean(X, axis=0)
            y_mean = np.mean(y)
            X_centered = X - X_mean
            y_centered = y - y_mean

            XT_X = X_centered.T @ X_centered
            #Do not regularize the intercept (identity matrix for features only)
            identity = np.eye(n_features)

            self.weights = np.linalg.inv(XT_X + self.alpha * identity) @ X_centered.T @ y_centered
            self.intercept = y_mean - (X_mean @ self.weights)

        elif self.penalty == 'lasso':
            #LASSO (L1) COORDINATE DESCENT ITERATION
            X_mean = np.mean(X, axis=0)
            y_mean = np.mean(y)
            X_centered = X - X_mean
            y_centered = y - y_mean

            #Initialize weights to zero
            self.weights = np.zeros(n_features)

            #Precompute column norms squared for speed optimization
            col_norms = np.sum(X_centered**2, axis=0)

            for iteration in range(self.max_iter):
                weights_old = self.weights.copy()

                for j in range(n_features):
                    #Compute partial residual excluding feature j
                    margin = X_centered[:, j] @ (y_centered - (X_centered @ self.weights) + self.weights[j] * X_centered[:, j])

                    #Apply Soft-Thresholding Operator: sign(rho) * max(0, |rho| - alpha)
                    if margin > self.alpha:
                        self.weights[j] = (margin - self.alpha) / col_norms[j]
                    elif margin < -self.alpha:
                        self.weights[j] = (margin + self.alpha) / col_norms[j]
                    else:
                        self.weights[j] = 0.0

                #Check convergence tolerance condition
                if np.sum(np.abs(self.weights - weights_old)) < self.tol:
                    break

            self.intercept = y_mean - (X_mean @ self.weights)
        else:
            raise ValueError("Unsupported penalty type. Choose 'ridge' or 'lasso'.")

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return (X @ self.weights) + self.intercept
if __name__ == "__main__":
    np.random.seed(42)
    #Generate 50 data samples with 4 distinct features
    X_sample = np.random.randn(50, 4)

    #Target formula uses ONLY feature 0 and feature 1. Features 2 & 3 are useless noise!
    y_sample = 10.0 + 5.0 * X_sample[:, 0] - 3.0 * X_sample[:, 1] + np.random.normal(0, 0.1, 50)

    #Evaluate Ridge Regression (L2)
    ridge_model = RegularizedLinearRegression(alpha=5.0, penalty='ridge')
    ridge_model.fit(X_sample, y_sample)

    #Evaluate Lasso Regression (L1)
    lasso_model = RegularizedLinearRegression(alpha=5.0, penalty='lasso')
    lasso_model.fit(X_sample, y_sample)

    #Output metric summaries side-by-side
    print(f"{'Parameter':<15} | {'True Value':<12} | {'Ridge (L2)':<12} | {'Lasso (L1)':<12}")
    print("-" * 60)
    print(f"{'Intercept':<15} | {10.0:<12.1f} | {ridge_model.intercept:<12.4f} | {lasso_model.intercept:<12.4f}")
    for i in range(4):
        true_w = 5.0 if i==0 else (-3.0 if i==1 else 0.0)
        print(f"{f'Weight {i}':<15} | {true_w:<12.1f} | {ridge_model.weights[i]:<12.4f} | {lasso_model.weights[i]:<12.4f}")


Parameter       | True Value   | Ridge (L2)   | Lasso (L1)  
------------------------------------------------------------
Intercept       | 10.0         | 9.9327       | 9.9953      
Weight 0        | 5.0          | 4.1022       | 4.7792      
Weight 1        | -3.0         | -2.6905      | -2.9166     
Weight 2        | 0.0          | -0.0186      | 0.0000      
Weight 3        | 0.0          | -0.0092      | 0.0000      


In [8]:
import numpy as np
from sklearn.linear_model import Ridge as SKRidge, Lasso as SKLasso

#CUSTOM REGULARIZED MODEL (From Previous Step)
class RegularizedLinearRegression:
    def __init__(self, alpha=1.0, penalty='ridge', max_iter=1000, tol=1e-4):
        self.alpha = alpha
        self.penalty = penalty.lower()
        self.max_iter = max_iter
        self.tol = tol
        self.weights = None
        self.intercept = 0.0

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).flatten()
        n_samples, n_features = X.shape

        if self.penalty == 'ridge':
            X_mean, y_mean = np.mean(X, axis=0), np.mean(y)
            X_centered, y_centered = X - X_mean, y - y_mean
            XT_X = X_centered.T @ X_centered
            identity = np.eye(n_features)
            self.weights = np.linalg.inv(XT_X + self.alpha * identity) @ X_centered.T @ y_centered
            self.intercept = y_mean - (X_mean @ self.weights)

        elif self.penalty == 'lasso':
            X_mean, y_mean = np.mean(X, axis=0), np.mean(y)
            X_centered, y_centered = X - X_mean, y - y_mean
            self.weights = np.zeros(n_features)
            col_norms = np.sum(X_centered**2, axis=0)

            #Scikit-learn normalizes the L1 objective function by dividing alpha by (2 * n_samples)
            #We scale our internal alpha down to exactly mirror their default coordinate descent math
            scaled_alpha = self.alpha * n_samples

            for iteration in range(self.max_iter):
                weights_old = self.weights.copy()
                for j in range(n_features):
                    margin = X_centered[:, j] @ (y_centered - (X_centered @ self.weights) + self.weights[j] * X_centered[:, j])
                    if margin > scaled_alpha:
                        self.weights[j] = (margin - scaled_alpha) / col_norms[j]
                    elif margin < -scaled_alpha:
                        self.weights[j] = (margin + scaled_alpha) / col_norms[j]
                    else:
                        self.weights[j] = 0.0
                if np.sum(np.abs(self.weights - weights_old)) < self.tol:
                    break
            self.intercept = y_mean - (X_mean @ self.weights)


#RUNNING THE HEAD-TO-HEAD VERIFICATION
if __name__ == "__main__":
    #Generate reproducible mock regression matrix
    np.random.seed(42)
    X_test = np.random.randn(100, 3)
    y_test = 5.0 + 2.5 * X_test[:, 0] - 1.2 * X_test[:, 1] + np.random.normal(0, 0.2, 100)

    alpha_val = 0.5

    #RIDGE COMPARISON
    scratch_ridge = RegularizedLinearRegression(alpha=alpha_val, penalty='ridge')
    scratch_ridge.fit(X_test, y_test)

    #Scikit-learn scales Ridge alpha differently depending on sample sizes.
    #To match standard normal analytical scaling, we match alpha scale directly.
    sk_ridge = SKRidge(alpha=alpha_val, solver='cholesky')
    sk_ridge.fit(X_test, y_test)

    ridge_weight_diff = np.abs(scratch_ridge.weights - sk_ridge.coef_)
    ridge_intercept_diff = np.abs(scratch_ridge.intercept - sk_ridge.intercept_)

    #LASSO COMPARISON
    scratch_lasso = RegularizedLinearRegression(alpha=alpha_val, penalty='lasso', max_iter=2000)
    scratch_lasso.fit(X_test, y_test)

    sk_lasso = SKLasso(alpha=alpha_val, max_iter=2000, tol=1e-4)
    sk_lasso.fit(X_test, y_test)

    lasso_weight_diff = np.abs(scratch_lasso.weights - sk_lasso.coef_)
    lasso_intercept_diff = np.abs(scratch_lasso.intercept - sk_lasso.intercept_)

    #Print comparison report
    print("SCIMIT-LEARN VS CUSTOM SCRATCH MODEL HEAD-TO-HEAD")
    print(f"Ridge Weight Max Absolute Error   : {np.max(ridge_weight_diff):.2e}")
    print(f"Ridge Intercept Absolute Error    : {ridge_intercept_diff:.2e}")
    print(f"Lasso Weight Max Absolute Error   : {np.max(lasso_weight_diff):.2e}")
    print(f"Lasso Intercept Absolute Error    : {lasso_intercept_diff:.2e}")


SCIMIT-LEARN VS CUSTOM SCRATCH MODEL HEAD-TO-HEAD
Ridge Weight Max Absolute Error   : 8.88e-16
Ridge Intercept Absolute Error    : 8.88e-16
Lasso Weight Max Absolute Error   : 2.29e-07
Lasso Intercept Absolute Error    : 1.93e-08


In [9]:
import numpy as np

class ResidualDiagnostics:
    @staticmethod
    def analyze(y_true, y_pred, X=None):
        """
        Runs analytical diagnostic tests on model residuals from scratch.
        """
        y_true = np.asarray(y_true).flatten()
        y_pred = np.asarray(y_pred).flatten()
        residuals = y_true - y_pred
        n_samples = len(residuals)

        results = {}

        #Linearity: Mean of residuals should be close to zero
        residual_mean = np.mean(residuals)
        results["linearity"] = {
            "residual_mean": residual_mean,
            "status": "PASS" if abs(residual_mean) < 1e-2 else "FAIL (Systematic Bias)"
        }

        #Independence: Durbin-Watson Test Heuristic
        #Range: 0 to 4. Metric near 2.0 = No autocorrelation.
        #Near 0 = Positive correlation; Near 4 = Negative correlation.
        diff_residuals = np.diff(residuals)
        dw_statistic = np.sum(diff_residuals ** 2) / np.sum(residuals ** 2)

        results["independence"] = {
            "durbin_watson_stat": dw_statistic,
            "status": "PASS" if 1.5 <= dw_statistic <= 2.5 else "WARNING (Autocorrelation Detected)"
        }

        #Normality: Skewness and Kurtosis Heuristics
        #Perfectly normal distribution has Skewness = 0 and Excess Kurtosis = 0
        mean_res = np.mean(residuals)
        std_res = np.std(residuals)

        if std_res > 0:
            skewness = np.mean(((residuals - mean_res) / std_res) ** 3)
            kurtosis = np.mean(((residuals - mean_res) / std_res) ** 4) - 3  # Excess Kurtosis
        else:
            skewness, kurtosis = 0, 0

        results["normality"] = {
            "skewness": skewness,
            "excess_kurtosis": kurtosis,
            "status": "PASS" if (abs(skewness) < 0.5 and abs(kurtosis) < 1.0) else "WARNING (Non-Normal Residuals)"
        }

        #Homoscedasticity: Goldfeld-Quandt Style Variance Split Heuristic
        #Split data in half by prediction size and compare variance scales
        sorted_indices = np.argsort(y_pred)
        res_sorted = residuals[sorted_indices]

        half = n_samples // 2
        var_first_half = np.var(res_sorted[:half])
        var_second_half = np.var(res_sorted[half:])

        #Avoid division by zero
        ratio = var_second_half / (var_first_half + 1e-8) if var_first_half > 0 else 1.0
        #If one half has double the variance of the other, homoscedasticity fails
        is_homo = 0.4 <= ratio <= 2.5

        results["homoscedasticity"] = {
            "variance_ratio": ratio,
            "status": "PASS" if is_homo else "FAIL (Heteroscedasticity / Unequal Variance)"
        }

        return results

    @staticmethod
    def print_report(diagnostic_results):
        """Prints a human-scannable summary of model health."""
        print(f"\n{'='*60}\n MODEL ASSUMPTION ASSESSMENTS (L.I.N.E)\n{'='*60}")

        l = diagnostic_results["linearity"]
        print(f" [L] Linearity        : {l['status']:<15} (Residual Mean: {l['residual_mean']:.6f})")

        i = diagnostic_results["independence"]
        print(f" [I] Independence     : {i['status']:<15} (Durbin-Watson Stat: {i['durbin_watson_stat']:.3f})")

        n = diagnostic_results["normality"]
        print(f" [N] Normality        : {n['status']:<15} (Skew: {n['skewness']:.3f}, Kurtosis: {n['excess_kurtosis']:.3f})")

        h = diagnostic_results["homoscedasticity"]
        print(f" [E] Equal Variance   : {h['status']:<15} (Variance Ratio Split: {h['variance_ratio']:.3f})")
if __name__ == "__main__":
    np.random.seed(42)
    X_line = np.linspace(1, 10, 200)

    #CASE 1: A healthy, well-behaved linear system
    y_healthy_true = 2.5 * X_line + np.random.normal(0, 1.0, size=200)
    #Simulate a good linear model fit
    y_healthy_pred = 2.5 * X_line

    results_healthy = ResidualDiagnostics.analyze(y_healthy_true, y_healthy_pred)
    print("\n>>> EVALUATING HEALTHY DATA RELATIONSHIP <<<")
    ResidualDiagnostics.print_report(results_healthy)

    #CASE 2: A broken system (Non-linear pattern + expanding variance noise)
    #The noise expands as X grows, which triggers heteroscedasticity
    expanding_noise = np.random.normal(0, 0.5 * X_line, size=200)
    y_broken_true = 0.5 * (X_line ** 2) + expanding_noise
    #Forcing a straight line fit onto a curved dataset
    y_broken_pred = 4.5 * X_line - 5.0

    results_broken = ResidualDiagnostics.analyze(y_broken_true, y_broken_pred)
    print("\n>>> EVALUATING BROKEN DATA RELATIONSHIP <<<")
    ResidualDiagnostics.print_report(results_broken)



>>> EVALUATING HEALTHY DATA RELATIONSHIP <<<

 MODEL ASSUMPTION ASSESSMENTS (L.I.N.E)
 [L] Linearity        : FAIL (Systematic Bias) (Residual Mean: -0.040771)
 [I] Independence     : PASS            (Durbin-Watson Stat: 2.090)
 [N] Normality        : PASS            (Skew: 0.132, Kurtosis: -0.002)
 [E] Equal Variance   : PASS            (Variance Ratio Split: 1.103)

>>> EVALUATING BROKEN DATA RELATIONSHIP <<<

 MODEL ASSUMPTION ASSESSMENTS (L.I.N.E)
 [L] Linearity        : FAIL (Systematic Bias) (Residual Mean: -0.990338)
 [I] Independence     : WARNING (Autocorrelation Detected) (Durbin-Watson Stat: 0.601)
 [N] Normality        : WARNING (Non-Normal Residuals) (Skew: 1.421, Kurtosis: 2.387)
 [E] Equal Variance   : FAIL (Heteroscedasticity / Unequal Variance) (Variance Ratio Split: 4.896)
